# Fourier analysis of synthetic seismic signals

Upstream research notebook from `taylor-kenyon/FluvialSeismology_Analysis`, `fourier_practice.ipynb`, commit `b4962248fdcfed970c0f5c64091472ab1933f2fc`. Saved outputs were cleared. Review inputs and any local paths before running.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.integrate import cumulative_trapezoid

# Create a single sine wave
duration = 2.0  # seconds
sampling_rate = 100  # Hz (samples per second)
t = np.arange(0, duration, 1/sampling_rate)

# Signal parameters
A = 1.0  # amplitude
f0 = 5.0  # frequency in Hz

# Generate signal
x = A * np.sin(2 * np.pi * f0 * t)

# Compute Fourier Transform
N = len(x)
X = fft(x)
freqs = fftfreq(N, 1/sampling_rate)

# Only plot positive frequencies (FFT is symmetric for real signals)
positive_freqs = freqs[:N//2]
X_magnitude = np.abs(X[:N//2]) * 2/N  # Normalize

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Time domain
ax1.plot(t, x, 'b-', linewidth=1.5)
ax1.set_xlabel('Time (s)', fontsize=12)
ax1.set_ylabel('Amplitude', fontsize=12)
ax1.set_title(f'Time Domain: Pure Sine Wave (f = {f0} Hz)', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, 1])  # Show first second

# Frequency domain
ax2.stem(positive_freqs, X_magnitude, basefmt=' ', linefmt='r-', markerfmt='ro')
ax2.set_xlabel('Frequency (Hz)', fontsize=12)
ax2.set_ylabel('Amplitude', fontsize=12)
ax2.set_title('Frequency Domain: Fourier Transform', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 20])

plt.tight_layout()
plt.show()

print(f"Signal: {A} * sin(2π * {f0} * t)")
print(f"Fourier Transform: Single peak at f = {f0} Hz")
print("\nKey Observation: A pure sine wave contains ONLY ONE frequency!")

In [ ]:
# Create a signal with three frequencies
f1, f2, f3 = 2, 5, 10  # Hz
A1, A2, A3 = 1.0, 0.7, 0.4  # Amplitudes

# Generate composite signal
x_multi = A1 * np.sin(2*np.pi*f1*t) + A2 * np.sin(2*np.pi*f2*t) + A3 * np.sin(2*np.pi*f3*t)

# Compute Fourier Transform
X_multi = fft(x_multi)
X_multi_magnitude = np.abs(X_multi[:N//2]) * 2/N

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Time domain
ax1.plot(t, x_multi, 'b-', linewidth=1.5, label='Composite signal')
ax1.plot(t, A1*np.sin(2*np.pi*f1*t), 'r--', alpha=0.5, linewidth=1, label=f'{f1} Hz component')
ax1.plot(t, A2*np.sin(2*np.pi*f2*t), 'g--', alpha=0.5, linewidth=1, label=f'{f2} Hz component')
ax1.plot(t, A3*np.sin(2*np.pi*f3*t), 'm--', alpha=0.5, linewidth=1, label=f'{f3} Hz component')
ax1.set_xlabel('Time (s)', fontsize=12)
ax1.set_ylabel('Amplitude', fontsize=12)
ax1.set_title('Time Domain: Sum of Three Sine Waves', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, 2])

# Frequency domain
ax2.stem(positive_freqs, X_multi_magnitude, basefmt=' ', linefmt='r-', markerfmt='ro')
ax2.axvline(f1, color='red', linestyle=':', alpha=0.5, label=f'{f1} Hz')
ax2.axvline(f2, color='green', linestyle=':', alpha=0.5, label=f'{f2} Hz')
ax2.axvline(f3, color='magenta', linestyle=':', alpha=0.5, label=f'{f3} Hz')
ax2.set_xlabel('Frequency (Hz)', fontsize=12)
ax2.set_ylabel('Amplitude', fontsize=12)
ax2.set_title('Frequency Domain: Fourier Transform Shows All Three Frequencies', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 15])

plt.tight_layout()
plt.show()

print(f"Signal components: {A1}*sin(2π*{f1}*t) + {A2}*sin(2π*{f2}*t) + {A3}*sin(2π*{f3}*t)")
print(f"Fourier Transform: Three peaks at {f1}, {f2}, and {f3} Hz")
print("\nKey Observation: Fourier transform DECOMPOSES the signal into its constituent frequencies!")
print("The amplitude of each peak tells us how much of that frequency is present.")

In [ ]:
# Integrate the synthetic signal with the trapezoidal rule.
dt = 1/sampling_rate

# Start with "acceleration"
acceleration = x_multi.copy()

# Integrate once to get "velocity"
velocity = cumulative_trapezoid(acceleration, dx=dt, initial=0)

# Integrate again to get "displacement"
displacement = cumulative_trapezoid(velocity, dx=dt, initial=0)

# Plot all three
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Acceleration
axes[0].plot(t, acceleration, 'b-', linewidth=1.5)
axes[0].set_ylabel('Acceleration\n(arbitrary units)', fontsize=11)
axes[0].set_title('Effect of Integration in Time Domain', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 2])

# Velocity
axes[1].plot(t, velocity, 'g-', linewidth=1.5)
axes[1].set_ylabel('Velocity\n(= ∫ acceleration dt)', fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 2])

# Displacement
axes[2].plot(t, displacement, 'r-', linewidth=1.5)
axes[2].set_ylabel('Displacement\n(= ∫ velocity dt)', fontsize=11)
axes[2].set_xlabel('Time (s)', fontsize=12)
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim([0, 2])

plt.tight_layout()
plt.show()

print("Notice how integration:")
print("1. Attenuates higher frequencies relative to lower frequencies")
print("2. Changes the phase (sine ↔ cosine)")
print("\nThis is why seismograms look different when showing acceleration vs. displacement!")

In [ ]:
# Compare the amplitude spectra of the three time-domain signals.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Compute FFTs
acceleration_amplitude = np.abs(fft(acceleration)[:N//2]) * 2/N
velocity_amplitude = np.abs(fft(velocity)[:N//2]) * 2/N
displacement_amplitude = np.abs(fft(displacement)[:N//2]) * 2/N

# Linear scale
ax1.plot(positive_freqs, acceleration_amplitude, 'b-', linewidth=2, label='Acceleration', alpha=0.7)
ax1.plot(positive_freqs, velocity_amplitude, 'g-', linewidth=2, label='Velocity', alpha=0.7)
ax1.plot(positive_freqs, displacement_amplitude, 'r-', linewidth=2, label='Displacement', alpha=0.7)
ax1.set_xlabel('Frequency (Hz)', fontsize=12)
ax1.set_ylabel('Amplitude (linear)', fontsize=12)
ax1.set_title('Frequency Spectra - Linear Scale', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, 15])

# Omit the zero-frequency bin in this view.
ax2.semilogy(positive_freqs[1:], acceleration_amplitude[1:], 'b-', linewidth=2, label='Acceleration', alpha=0.7)
ax2.semilogy(positive_freqs[1:], velocity_amplitude[1:], 'g-', linewidth=2, label='Velocity', alpha=0.7)
ax2.semilogy(positive_freqs[1:], displacement_amplitude[1:], 'r-', linewidth=2, label='Displacement', alpha=0.7)
ax2.set_xlabel('Frequency (Hz)', fontsize=12)
ax2.set_ylabel('Amplitude (log scale)', fontsize=12)
ax2.set_title('Frequency Spectra - Log Scale', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 15])

plt.tight_layout()
plt.show()

print("Key Observations:")
print("1. Integration emphasizes lower frequencies relative to higher frequencies")
print("2. The plotted amplitudes describe different quantities and have different units")
print("3. Integrating a sinusoid scales its amplitude by 1/(2πf), for nonzero f")
print("\nThis is why:")
print("- Strong motion seismometers record ACCELERATION (captures high-freq shaking)")
print("- Broadband seismometers often show VELOCITY or DISPLACEMENT (long-period waves)")

In [ ]:
# Demonstrate spectral leakage with a non-integer number of cycles
# A partial cycle creates a discontinuity when the FFT repeats the window.

duration_leak = 2.0
t_leak = np.arange(0, duration_leak, 1/sampling_rate)
f_leak = 5.3  # The two-second window contains 10.6 cycles.

# Pure sine wave - but doesn't complete integer cycles
x_leak = np.sin(2 * np.pi * f_leak * t_leak)

# Compute spectrum
N_leak = len(x_leak)
X_leak = fft(x_leak)
freqs_leak = fftfreq(N_leak, 1/sampling_rate)
X_leak_mag = np.abs(X_leak[:N_leak//2]) * 2/N_leak

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Time domain - show the discontinuity
axes[0, 0].plot(t_leak, x_leak, 'b-', linewidth=1.5)
axes[0, 0].scatter([t_leak[0], t_leak[-1]], [x_leak[0], x_leak[-1]], 
                   s=100, c='red', zorder=5, label='Start/End points')
axes[0, 0].set_xlabel('Time (s)', fontsize=11)
axes[0, 0].set_ylabel('Amplitude', fontsize=11)
axes[0, 0].set_title('Time Domain: Sine Wave (5.3 Hz)', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Show what FFT "sees" when it wraps around
axes[0, 1].plot(t_leak[-20:], x_leak[-20:], 'b-', linewidth=2, label='End of signal')
axes[0, 1].plot(t_leak[:20] + duration_leak, x_leak[:20], 'g--', linewidth=2, label='Start (wrapped)')
axes[0, 1].axvline(duration_leak, color='red', linestyle=':', linewidth=2, label='Discontinuity!')
axes[0, 1].set_xlabel('Time (s)', fontsize=11)
axes[0, 1].set_ylabel('Amplitude', fontsize=11)
axes[0, 1].set_title('What FFT "Sees": Periodic Extension', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Frequency domain - show the leakage
axes[1, 0].stem(freqs_leak[:N_leak//2], X_leak_mag, basefmt=' ')
axes[1, 0].axvline(f_leak, color='red', linestyle='--', alpha=0.5, label=f'True freq = {f_leak} Hz')
axes[1, 0].set_xlabel('Frequency (Hz)', fontsize=11)
axes[1, 0].set_ylabel('Amplitude', fontsize=11)
axes[1, 0].set_title('Frequency Domain: Spectral Leakage!', fontsize=12, fontweight='bold')
axes[1, 0].set_xlim([0, 20])
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Zoom in to see the spreading
axes[1, 1].plot(freqs_leak[:N_leak//2], X_leak_mag, 'b-', linewidth=1.5)
axes[1, 1].axvline(f_leak, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'True freq = {f_leak} Hz')
axes[1, 1].set_xlabel('Frequency (Hz)', fontsize=11)
axes[1, 1].set_ylabel('Amplitude', fontsize=11)
axes[1, 1].set_title('Zoomed: Energy Spreads to Adjacent Frequencies', fontsize=12, fontweight='bold')
axes[1, 1].set_xlim([3, 8])
axes[1, 1].set_ylim([0, 0.6])
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Start value: {x_leak[0]:.4f}")
print(f"End value: {x_leak[-1]:.4f}")
print(f"Discontinuity: {abs(x_leak[-1] - x_leak[0]):.4f}")
print("\nThe 5.3 Hz tone contributes energy across multiple FFT bins.")
print("This spreading across bins is spectral leakage from the finite window.")

In [ ]:
# Apply taper to the same signal and compare
from scipy.signal import windows

# The Hann taper brings both endpoints to zero.
hann_taper = windows.hann(N_leak)

# Apply taper
x_tapered = x_leak * hann_taper

# Compute spectrum of tapered signal
X_tapered = fft(x_tapered)
X_tapered_mag = np.abs(X_tapered[:N_leak//2]) * 2/N_leak

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Time domain - no taper
axes[0, 0].plot(t_leak, x_leak, 'b-', linewidth=1.5, label='Original')
axes[0, 0].scatter([t_leak[0], t_leak[-1]], [x_leak[0], x_leak[-1]], 
                   s=100, c='red', zorder=5)
axes[0, 0].set_ylabel('Amplitude', fontsize=11)
axes[0, 0].set_title('WITHOUT Taper: Discontinuity at Edges', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xlim([0, 2])

# Time domain - with taper
axes[0, 1].plot(t_leak, x_leak, 'b--', alpha=0.3, linewidth=1, label='Original')
axes[0, 1].plot(t_leak, x_tapered, 'g-', linewidth=1.5, label='Tapered')
axes[0, 1].plot(t_leak, hann_taper, 'r:', linewidth=2, alpha=0.7, label='Taper function')
axes[0, 1].set_ylabel('Amplitude', fontsize=11)
axes[0, 1].set_title('WITH Taper: Smooth to Zero at Edges', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xlim([0, 2])

# Frequency domain - no taper
axes[1, 0].plot(freqs_leak[:N_leak//2], X_leak_mag, 'b-', linewidth=1.5)
axes[1, 0].axvline(f_leak, color='red', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Frequency (Hz)', fontsize=11)
axes[1, 0].set_ylabel('Amplitude', fontsize=11)
axes[1, 0].set_title('Spectrum WITHOUT Taper: Spectral Leakage', fontsize=12, fontweight='bold')
axes[1, 0].set_xlim([3, 8])
axes[1, 0].set_ylim([0, 0.6])
axes[1, 0].grid(True, alpha=0.3)

# Frequency domain - with taper
axes[1, 1].plot(freqs_leak[:N_leak//2], X_tapered_mag, 'g-', linewidth=1.5)
axes[1, 1].axvline(f_leak, color='red', linestyle='--', alpha=0.5, label=f'{f_leak} Hz')
axes[1, 1].set_xlabel('Frequency (Hz)', fontsize=11)
axes[1, 1].set_ylabel('Amplitude', fontsize=11)
axes[1, 1].set_title('Spectrum WITH Taper: Leakage Suppressed!', fontsize=12, fontweight='bold')
axes[1, 1].set_xlim([3, 8])
axes[1, 1].set_ylim([0, 0.6])
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("With the Hann taper:")
print("  - Energy concentrated near true frequency (5.3 Hz)")
print("  - Minimal spectral leakage to adjacent frequencies")
print("  - Peak is broader but cleaner")
print("\nTrade-off:")
print("  - Slight frequency resolution loss (broader peak)")
print("  - Reduced leakage away from the peak")

In [ ]:
# Create a signal segment that doesn't start/end at zero
t_filt = np.arange(0, 3.0, 1/sampling_rate)
# Add large low-frequency component + signal of interest + DC offset
# The 4–6 Hz filter attenuates the large 0.5 Hz component. Compare the edge transients with and without tapering.
x_filt = (2.0*np.sin(2*np.pi*0.5*t_filt) +      # Large low-freq (will be filtered out)
          np.sin(2*np.pi*2*t_filt) +             # Below passband
          0.7*np.sin(2*np.pi*5*t_filt) +         # In passband (kept)
          0.3*np.sin(2*np.pi*10*t_filt) +        # Above passband
          1.0)  # DC offset

# Design bandpass filter (keep 4-6 Hz)
from scipy.signal import butter, filtfilt
b, a = butter(4, [4, 6], btype='band', fs=sampling_rate)

# Filter WITHOUT tapering
x_filt_notaper = filtfilt(b, a, x_filt)

# Filter WITH tapering
taper_filt = windows.tukey(len(x_filt), alpha=0.1)  # Tukey taper across 10% of the full window, split across both edges
x_filt_tapered_input = x_filt * taper_filt
x_filt_withtaper = filtfilt(b, a, x_filt_tapered_input)

# Plot comparison
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Original signal
axes[0].plot(t_filt, x_filt, 'k-', linewidth=1.5)
axes[0].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[0].fill_between([0, 0.3], -4, 4, alpha=0.2, color='red', label='Edge regions')
axes[0].fill_between([2.7, 3.0], -4, 4, alpha=0.2, color='red')
axes[0].set_ylabel('Amplitude', fontsize=11)
axes[0].set_title('Original Signal (Large 0.5 Hz + 5 Hz target + DC)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 3])
axes[0].set_ylim([-4, 4])

# Filtered WITHOUT taper
axes[1].plot(t_filt, x_filt_notaper, 'b-', linewidth=1.5)
axes[1].axhline(0, color='gray', linestyle=':', linewidth=1)
# Highlight edge artifacts
axes[1].axvspan(0, 0.8, alpha=0.3, color='red', label='HUGE edge ringing!')
axes[1].axvspan(2.2, 3.0, alpha=0.3, color='red')
axes[1].set_ylabel('Amplitude', fontsize=11)
axes[1].set_title('Filtered (4-6 Hz) WITHOUT Tapering: Massive Ringing!', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 3])
axes[1].set_ylim([-2.0, 2.0])

# Filtered WITH taper
axes[2].plot(t_filt, x_filt_withtaper, 'g-', linewidth=1.5)
axes[2].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[2].set_ylabel('Amplitude', fontsize=11)
axes[2].set_xlabel('Time (s)', fontsize=12)
axes[2].set_title('Filtered (4-6 Hz) WITH Tapering: Clean Edges', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim([0, 3])
axes[2].set_ylim([-2.0, 2.0])

plt.tight_layout()
plt.show()

print("Edge artifacts WITHOUT tapering:")
print(f"  Max amplitude near start: {np.max(np.abs(x_filt_notaper[:50])):.4f}")
print(f"  Max amplitude near end: {np.max(np.abs(x_filt_notaper[-50:])):.4f}")
print("\nEdge behavior WITH tapering:")
print(f"  Max amplitude near start: {np.max(np.abs(x_filt_withtaper[:50])):.4f}")
print(f"  Max amplitude near end: {np.max(np.abs(x_filt_withtaper[-50:])):.4f}")
print(f"\nPeak amplitude in the first 100 samples is {np.max(np.abs(x_filt_notaper[:100]))/np.max(np.abs(x_filt_withtaper[:100])):.1f} times as large without tapering.")
print("Filter transients at the window edges can resemble seismic arrivals.")


In [ ]:
# Differentiate a synthetic signal with a discontinuity at the periodic boundary.
t_toy = np.arange(0, 2.0, 1/sampling_rate)
# FFT differentiation multiplies each coefficient by iω, amplifying high-frequency effects at the boundary.
x_toy = 3.0*np.sin(2*np.pi*0.7*t_toy) + np.sin(2*np.pi*3*t_toy) + 2.5

# Differentiate in frequency domain (multiply by iω)
X_toy = np.fft.fft(x_toy)
freqs_toy = np.fft.fftfreq(len(x_toy), 1/sampling_rate)
omega = 2*np.pi*freqs_toy

# WITHOUT tapering
dx_notaper = np.real(np.fft.ifft(1j * omega * X_toy))

# WITH tapering
taper_toy = windows.hann(len(x_toy))
x_toy_tapered = x_toy * taper_toy
X_toy_tapered = np.fft.fft(x_toy_tapered)
dx_tapered = np.real(np.fft.ifft(1j * omega * X_toy_tapered))

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Row 1: Original signals
axes[0,0].plot(t_toy, x_toy, 'k-', linewidth=1.5, label='Untapered')
axes[0,0].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[0,0].axvspan(0, 0.2, alpha=0.2, color='red')
axes[0,0].axvspan(1.8, 2.0, alpha=0.2, color='red')
axes[0,0].set_ylabel('Amplitude', fontsize=11)
axes[0,0].set_title('Original Signal (Large 0.7 Hz + 3 Hz + DC = 2.5)', fontsize=11, fontweight='bold')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)
axes[0,0].set_xlim([0, 2])
axes[0,0].set_ylim([-1, 6])

axes[0,1].plot(t_toy, x_toy_tapered, 'g-', linewidth=1.5, label='Tapered')
axes[0,1].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[0,1].set_ylabel('Amplitude', fontsize=11)
axes[0,1].set_title('Tapered Signal (smooth edges)', fontsize=11, fontweight='bold')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)
axes[0,1].set_xlim([0, 2])
axes[0,1].set_ylim([-1, 6])

# Row 2: After differentiation
axes[1,0].plot(t_toy, dx_notaper, 'r-', linewidth=1.5)
axes[1,0].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[1,0].axvspan(0, 0.4, alpha=0.3, color='red', label='EXTREME artifacts!')
axes[1,0].axvspan(1.6, 2.0, alpha=0.3, color='red')
axes[1,0].set_ylabel('Amplitude', fontsize=11)
axes[1,0].set_xlabel('Time (s)', fontsize=12)
axes[1,0].set_title('After Differentiation WITHOUT Taper: MASSIVE Artifacts!', fontsize=11, fontweight='bold', color='red')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_xlim([0, 2])
axes[1,0].set_ylim([-600, 600])

axes[1,1].plot(t_toy, dx_tapered, 'g-', linewidth=1.5)
axes[1,1].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[1,1].set_ylabel('Amplitude', fontsize=11)
axes[1,1].set_xlabel('Time (s)', fontsize=12)
axes[1,1].set_title('After Differentiation WITH Taper: Clean Result', fontsize=11, fontweight='bold', color='green')
axes[1,1].grid(True, alpha=0.3)
axes[1,1].set_xlim([0, 2])
axes[1,1].set_ylim([-600, 600])

plt.tight_layout()
plt.show()

print("Maximum artifact magnitude WITHOUT tapering:")
print(f"  Near start: {np.max(np.abs(dx_notaper[:50])):.1f}")
print(f"  Near end: {np.max(np.abs(dx_notaper[-50:])):.1f}")
print(f"  Central region: {np.max(np.abs(dx_notaper[len(dx_notaper)//3:2*len(dx_notaper)//3])):.1f}")
print("\nMaximum magnitude WITH tapering:")
print(f"  Near start: {np.max(np.abs(dx_tapered[:50])):.1f}")
print(f"  Near end: {np.max(np.abs(dx_tapered[-50:])):.1f}")
print(f"  Central region: {np.max(np.abs(dx_tapered[len(dx_tapered)//3:2*len(dx_tapered)//3])):.1f}")
print(f"\nThe untapered edge peak is {np.max(np.abs(dx_notaper[:50]))/np.max(np.abs(dx_tapered[len(dx_tapered)//3:2*len(dx_tapered)//3])):.0f} times the peak in the middle third of the tapered derivative.")
print("The edge transient dominates this example without tapering.")
print("\nInspect the edges when integrating or differentiating a finite window.")

In [ ]:
print("""Tapering finite windows\n
The FFT treats the input as one period of a repeating signal. If the end and
start do not join smoothly, power can leak into other frequency bins.\n
Consider tapering before computing spectra, filtering, differentiation,
integration, or cross-correlation. A taper downweights edge samples and can
reduce boundary artifacts. In a spectrum it also broadens peaks.""")

Unfinished upstream draft retained for reference.

```text
the_volicit_ the mnf ejrejf ejythe 
```